<a href="https://colab.research.google.com/github/nivethithanm/torchcode-solutions/blob/main/P0_06_autograd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# P0-06: Autograd Fundamentals

**Difficulty**: 🟢 Easy

**Objective**: Understand how PyTorch tracks computations and computes gradients. This is the engine behind ALL training.

In [1]:
import torch
import torch.nn as nn

## Task 1: Scalar Gradients

In [3]:
x = torch.tensor(3.0, requires_grad=True)

# TODO: Compute y = x^2 + 2x + 1
y = x**2 + 2*x + 1  # YOUR CODE

# TODO: Compute gradient dy/dx
x.grad = 2*x + 2 # (one line)

# The gradient of x^2 + 2x + 1 at x=3 is 2*3 + 2 = 8
assert x.grad == 8.0, f'Expected 8.0, got {x.grad}'
print('✅ Task 1 passed!')

✅ Task 1 passed!


## Task 2: Vector Gradients & Zero Grad

In [6]:
w = torch.randn(3, requires_grad=True)
x = torch.tensor([1.0, 2.0, 3.0])

# Forward pass 1
loss1 = (w * x).sum()
loss1.backward()
grad1 = w.grad.clone()

# TODO: What happens if we do another backward WITHOUT zeroing grad?
loss2 = (w * x).sum()
loss2.backward()
grad2 = w.grad.clone()

# TODO: Explain in a comment what you observe
print(f'After 1st backward: {grad1}')
print(f'After 2nd backward (no zero): {grad2}')
# YOUR EXPLANATION HERE:
# ...

# TODO: Now zero the grad and redo — verify it matches grad1
w.grad.zero_()
loss3 = (w * x).sum()
loss3.backward()
grad3 = w.grad.clone()
assert torch.allclose(grad1, grad3)

print('✅ Task 2 passed!')

After 1st backward: tensor([1., 2., 3.])
After 2nd backward (no zero): tensor([2., 4., 6.])
✅ Task 2 passed!


## Task 3: no_grad & detach

Critical for inference and when you don't want gradient tracking.

In [13]:
w = torch.randn(5, requires_grad=True)

# TODO: Compute output = w * 2 without tracking gradients (two ways)
# Way 1: torch.no_grad() context
with torch.no_grad():
    out1 = w * 2  # YOUR CODE

# Way 2: .detach()
out2 = w * 2  # YOUR CODE
out2 = out2.detach()

assert not out1.requires_grad
assert not out2.requires_grad
print('✅ Task 3 passed!')

✅ Task 3 passed!


## Task 4: Manual SGD Training Loop

Train a single linear layer to learn y = 2x + 1

In [30]:
torch.manual_seed(42)
# Data
x = torch.linspace(0, 1, 50).unsqueeze(1)  # (50, 1)
y_true = 2 * x + 1 + 0.1 * torch.randn_like(x)  # noisy target

# Parameters
w = torch.randn(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
lr = 0.2

# TODO: Training loop — 100 steps of SGD
for step in range(100):
    # Forward: y_pred = x * w + b
    # Loss: MSE
    # Backward
    # Update w and b (inside no_grad!)
    # Zero gradients
    y_pred = x * w + b
    loss = ((y_pred - y_true)**2).mean()
    loss.backward()
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad
        w.grad.zero_()
        b.grad.zero_()

print(f'Learned: y = {w.item():.2f}x + {b.item():.2f}')
print(f'Target:  y = 2.00x + 1.00')
assert abs(w.item() - 2.0) < 0.2, f'w should be ~2.0, got {w.item():.2f}'
assert abs(b.item() - 1.0) < 0.2, f'b should be ~1.0, got {b.item():.2f}'
print('✅ Task 4 passed!')

Learned: y = 1.90x + 1.06
Target:  y = 2.00x + 1.00
✅ Task 4 passed!
